# Solution 02: Sentiment Analysis

In [1]:
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer
import torch, json, os

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "product_reviews.json")) as f:
    reviews = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Loaded {len(reviews)} reviews")

✓ Loaded 15 reviews


## Part 1: Load Pipeline and Classify a Review

In [2]:
model = GLiClassModel.from_pretrained("knowledgator/gliclass-edge-v3.0")
tokenizer = AutoTokenizer.from_pretrained("knowledgator/gliclass-edge-v3.0", add_prefix_space=True)
pipeline = ZeroShotClassificationPipeline(
    model, tokenizer, classification_type='single-label', device=device
)

test_review = "Absolutely love this headphone! The sound quality is crystal clear and the noise cancellation is incredible. Best purchase I have made all year."
sentiment_labels = ["positive", "negative", "neutral"]
sentiment_result = pipeline(test_review, sentiment_labels)[0]

assert sentiment_result[0]['label'] == 'positive'
print(f"✓ Classified as '{sentiment_result[0]['label']}' ({sentiment_result[0]['score']:.2f})")

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:02<00:00,  2.26s/it]

100%|██████████| 1/1 [00:02<00:00,  2.26s/it]

✓ Classified as 'positive' (0.94)


## Part 2: Implement `analyze_sentiment`

In [3]:
def analyze_sentiment(pipeline, texts, labels):
    """Batch sentiment classification. Returns list of {text, label, score}."""
    results = pipeline(texts, labels, batch_size=8)
    return [
        {"text": text, "label": res[0]['label'], "score": res[0]['score']}
        for text, res in zip(texts, results)
    ]


sample_texts = [reviews[0]['text'], reviews[1]['text'], reviews[2]['text']]
sample_results = analyze_sentiment(pipeline, sample_texts, sentiment_labels)

assert len(sample_results) == 3
for res in sample_results:
    assert {'text', 'label', 'score'} <= res.keys()
assert sample_results[0]['label'] == 'positive'
assert sample_results[1]['label'] == 'negative'
print("✓ analyze_sentiment works")
for res in sample_results:
    print(f"  [{res['label']:8}] ({res['score']:.2f}) {res['text'][:60]}...")

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  2.49it/s]

100%|██████████| 1/1 [00:00<00:00,  2.49it/s]

✓ analyze_sentiment works
  [positive] (0.88) Absolutely love this headphone! The sound quality is crystal...
  [negative] (1.00) Complete waste of money. The charging cable broke after two ...
  [negative] (0.93) It works fine for the price. Nothing exceptional — the scree...


## Part 3: Accuracy on Full Dataset

In [4]:
texts = [r['text'] for r in reviews]
results = analyze_sentiment(pipeline, texts, sentiment_labels)
correct = sum(1 for r, rev in zip(results, reviews) if r['label'] == rev['sentiment'])
accuracy = correct / len(reviews)

assert accuracy >= 0.70, f"Accuracy {accuracy:.0%} < 70%"
print(f"✓ Accuracy: {accuracy:.0%} ({correct}/{len(reviews)})")
for res, rev in zip(results, reviews):
    match = "✓" if res['label'] == rev['sentiment'] else "✗"
    print(f"  {match} [{rev['sentiment']:8}] predicted={res['label']:8} ({res['score']:.2f})")

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 61.25it/s]

✓ Accuracy: 80% (12/15)
  ✓ [positive] predicted=positive (0.88)
  ✓ [negative] predicted=negative (1.00)
  ✗ [neutral ] predicted=negative (0.93)
  ✓ [positive] predicted=positive (0.86)
  ✓ [negative] predicted=negative (0.84)
  ✓ [neutral ] predicted=neutral  (0.98)
  ✓ [positive] predicted=positive (0.95)
  ✓ [negative] predicted=negative (1.00)
  ✓ [neutral ] predicted=neutral  (0.57)
  ✓ [positive] predicted=positive (0.95)
  ✓ [negative] predicted=negative (0.62)
  ✗ [neutral ] predicted=positive (0.51)
  ✓ [positive] predicted=positive (0.58)
  ✓ [negative] predicted=negative (1.00)
  ✗ [neutral ] predicted=negative (0.96)
